<a href="https://colab.research.google.com/github/fariha-experiments/H-Clinical-Guideline-RAG/blob/google-colab-notebook-v1/clinical_guideline_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing libraries needed

In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━

Library	Restaurant Analogy

---


pypdf	Waiter who brings the ingredients into the kitchen

Text Splitter	Prep chef who chops ingredients into usable pieces

Sentence Transformers	Chef who identifies the flavor profile of each ingredient

ChromaDB	Pantry that organizes ingredients by
flavor so similar ones are stored together

LangChain	Head chef coordinating the whole kitchen\

---



In [ ]:
import langchain
import langchain_community
import langchain_text_splitters
import chromadb
import sentence_transformers
import pypdf

# Also import necessary components from langchain
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

/tmp/ipykernel_3028/3098538601.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [ ]:
#/*from langchain.embeddings import HuggingFaceEmbeddings*/

from langchain.embeddings import HuggingFaceEmbeddings

---
Error:ImportError: cannot import name 'HuggingFaceEmbeddings' from 'langchain.embeddings'

Fix: below


In [ ]:
!pip install -q langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

Originally, LangChain bundled everything together(vectors,db,..)
as it grew, it got too hard to maintain. so it was split into:

langchain              → Core framework

langchain-community    → Community-maintained integrations

langchain-huggingface  → Hugging Face-specific code

langchain-openai       → OpenAI-specific code

langchain-text-splitters → Text splitting utilities

In [ ]:
!pip list | grep langchain

langchain                                1.3.9
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.4.7
langchain-huggingface                    1.2.2
langchain-protocol                       0.0.17
langchain-text-splitters                 1.1.2


first you load the pdf after adding it in your files. == loader

then, you

In [ ]:
loader = PyPDFLoader("/content/Evidence-Based-Guidelines-2023.pdf")
documents = loader.load()

print(f"Number of documents loaded: {len(documents)}")
print(documents[0].page_content[:700])

Number of documents loaded: 261
International Evidence-based 
Guideline for the assessment 
and management of 
polycystic ovary syndrome 
2023


documents[0] is the first page, and documents[1] is the second, and so on. This means you have successfully loaded the content of your PDF page by page.


each page of pdf is loaded as a list item.
so document is a list.
document[1].page_content = pdf page 1 content

---



> *documents[0].page_content, you are getting the string of text from the first page, and documents[0].metadata gives you a dictionary of details about that page*



In [ ]:
print(documents[0].page_content[:1000])

International Evidence-based 
Guideline for the assessment 
and management of 
polycystic ovary syndrome 
2023


In [ ]:
print(f"Actual length of the first page content: {len(documents[0].page_content)} characters")

Actual length of the first page content: 110 characters


In [ ]:
print(f"Content of the second page (documents[1].page_content[:500]):\n{documents[1].page_content[:500]}")
print(f"Actual length of the second page content: {len(documents[1].page_content)} characters")

Content of the second page (documents[1].page_content[:500]):
Disclaimer 
The Centre for Research Excellence in Women’s Health in Reproductive 
Life (CRE WHiRL), worked in partnership with the American Society of 
Reproductive Medicine (ASRM), the Endocrine Society, the European Society 
of Endocrinology and the European Society of Human Reproduction and 
Embryology (ESHRE) and in collaboration with professional societies and 
consumer advocacy groups internationally. These evidence-based guidelines 
were developed to provide evidence-based recommendations
Actual length of the second page content: 5537 characters


text spilt:

The chunk_size of 800 characters acts as an upper limit for each chunk.

Since documents[0] (your first page) only has 110 characters, and this is less than your specified chunk_size of 800, the splitter doesn't try to force it to be 800 characters. Instead, it takes the entire content of that first page as the first chunk

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
chunks = text_splitter.split_documents(documents)

print(f"Number of original documents: {len(documents)}")
print(f"Number of chunks created after splitting: {len(chunks)}")
print("\n--- First chunk content (first 500 characters) ---")
print(chunks[0].page_content[:500])
print(f"Length of the first chunk: {len(chunks[0].page_content)} characters")

Number of original documents: 261
Number of chunks created after splitting: 1154

--- First chunk content (first 500 characters) ---
International Evidence-based 
Guideline for the assessment 
and management of 
polycystic ovary syndrome 
2023
Length of the first chunk: 110 characters


The RecursiveCharacterTextSplitter processes each Document (which, in our case, is each page's page_content) independently.

Here's how it works:

Iterates through documents: It goes through your documents list one by one.

Checks page_content length: For each document.page_content it encounters, it looks at its length.

If page_content < chunk_size: , then that entire page's content becomes a single chunk. It doesn't try to fill it up or combine it with other pages.

If page_content > chunk_size:  then the splitter will break that single page's content into multiple chunks, each around the chunk_size limit (and with the specified chunk_overlap).
So, in essence, it's not forcing every chunk to be 800 characters. It's ensuring that no chunk exceeds 800 characters, and it will keep natural breaks (like entire short pages) intact when possible.

In [ ]:
print(chunks[10].page_content)

1 International Evidence-based Guideline for the assessment and management of polycystic ovary syndrome 2023 
 
 
 
Acknowledgements 
We gratefully acknowledge the contribution of our engaged, funding, partner and collaborating organisations: 
1 The Australian National Health and Medical Research Council (NHMRC) through the funded Centre 
for Research Excellence in Women’s Health in Reproductive Life (CRE WHiRL) (APP1171592) and Centre for 
Research Excellence in Polycystic Ovary Syndrome (CRE PCOS) (APP1078444) and the members of this 
Centre who led and coordinated this international guideline effort  
2 Our partner organisations which co-funded the guideline: 
 American Society for Reproductive Medicine (ASRM) 
 Endocrine Society (ENDO) 
 European Society of Endocrinology (ESE)


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="pcos_guidelines"
)

doc.metadata is a dictionary that contains additional information about the document, such as the page number. .get('page', 'Unknown') is a safe way to access a dictionary value.
'Unknown': This is the default value that get() returns if the key 'page' is not found in the doc.metadata dictionary. This prevents a KeyError if some documents happen to not have a 'page' entry in their metadata, making your code more robust.

In [ ]:
question = "What are the symptoms of PCOS?"

results = db.similarity_search(question, k=3)

for i, doc in enumerate(results):
    print("=" * 80)
    print(f"Result {i+1}")
    print(f"Page: {doc.metadata.get('page', 'Unknown')}")
    print()
    print(doc.page_content)
    print()

Result 1
Page: 8

The new guidelines used best practice to bring together evidence, expert perspectives and the preferences 
of women with PCOS. They note that PCOS can be diagnosed using specific signs, symptoms, and blood tests. 
The recommendations also simplify the tests needed for diagnosis. Once diagnosed, their doctors and healthcare 
professionals should identify and take care of the many different aspects of health and increased health risks 
affected by PCOS. This includes recognising increased risks in reproductive health such as reduced fertility, 
metabolism such as diabetes, heart health, skin health, sleep, and mental health such as depression.  
Doctors need to be aware of increased risks and check and intervene to prevent and limit these.

Result 2
Page: 9

improved patient experience and health outcomes for the one in ten women worldwide with PCOS. 
 
Context and background 
Polycystic ovary syndrome (PCOS) is a significant public health issue with endocrine, reproduc

##Stage 2 : Integrating llm

In [ ]:
!pip install -q google-generativeai

In [ ]:
import google.generativeai as genai

In [ ]:
from google.colab import userdata


genai.configure(api_key=userdata.get('gemini_key'))

In [ ]:
model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
#results = db.similarity_search(question, k=3)

caught AI error which repeated the same code again. At first, suggested changing variable name coz I didn't want to override, then realised it's a mistake(building integration flow separately so not huge)

In [ ]:
context = "\n\n".join(doc.page_content for doc in results)

In [ ]:
prompt = f"""
You are a medical assistant.

Answer ONLY using the context below.

If the answer is not present, say:
'I couldn't find this information in the provided guideline.'

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
response = model.generate_content(prompt)

print(response.text)

Women with PCOS have diverse features including:
*   **Psychological:** anxiety, depression, sleep and eating disorders
*   **Dermatologic:** hirsutism, acanthosis nigricans and acne
*   **Reproductive:** irregular menstrual cycles, infertility, endometrial cancer and pregnancy complications
*   **Metabolic:** insulin resistance, metabolic...


# Q4

Question:
What are the symptoms of PCOS?

Topic Retrieval:
✅ Good

Chunk Quality:
⭐⭐⭐⭐☆

Answerability:
⭐⭐⭐⭐☆

Key Observation:
Result 2 is the strongest answer and should ideally rank above Result 1.

## Evaluation Log

### Q1
Question: What are the diagnostic criteria for PCOS?

Expected:
Diagnosis section (Rotterdam criteria)

Retrieved:
Page 63 (AMH)
Page 60 (Ultrasound)
Page 60 (Rotterdam)

Inference:
Broad question. Retrieved diagnosis-related evidence but not the concise definition.

### Q2
Question: What is AMH?

Expected:
AMH section

Retrieved:
Page 27
Page 64
Page 64

Inference:
Excellent retrieval. Specific concepts retrieve much better than umbrella concepts.

## Q3

### Question
What is hyperandrogenism?

### Expected Retrieval
A chunk defining hyperandrogenism, ideally explaining:
- Clinical hyperandrogenism
- Biochemical hyperandrogenism
- Associated clinical features (e.g. hirsutism, acne, androgenic alopecia)

---

### Retrieved Chunks

#### Result 1 (Page 228)
**Topic Retrieval:** ✅ Excellent
**Chunk Quality:** ✅ Excellent

**Reasoning**
- Directly answers the question.
- Clearly distinguishes clinical vs biochemical hyperandrogenism.
- Self-contained and understandable without additional context.

---

#### Result 2 (Page 59)
**Topic Retrieval:** 🟡 Relevant
**Chunk Quality:** ❌ Poor

**Reasoning**
- Correct topic (clinical hyperandrogenism).
- Starts midway through an existing paragraph.
- Lacks surrounding context.
- Does not define or explain hyperandrogenism.
- Limited standalone value for a downstream LLM.

---

#### Result 3 (Page 67)
**Topic Retrieval:** 🟡 Relevant
**Chunk Quality:** ❌ Poor

**Reasoning**
- Mentions persistent hyperandrogenism.
- Discussion assumes prior context.
- Not useful for answering "What is hyperandrogenism?"
- Better suited for questions about follow-up or persistence.

---

### Overall Evaluation

**Topic Retrieval:** ⭐⭐⭐⭐☆ (4/5)

The retriever correctly identified the semantic topic. All retrieved chunks discuss hyperandrogenism.

**Chunk Quality:** ⭐⭐☆☆☆ (2/5)

Only one retrieved chunk is independently useful. The remaining chunks appear to be split mid-discussion and lose important context.

---

### Hypothesis

Current retrieval quality is limited more by **chunk quality** than by **semantic retrieval**.

Evidence suggests:
- Embeddings successfully identify the correct topic.
- Character-based chunking sometimes produces incomplete chunks that reduce answer quality.

This motivates future experiments with:
- section/heading-based chunking
- paragraph-aware chunking
- semantic chunking

 results = db.similarity_search_with_score(
    question,
    k=3
)


for doc, score in results:
    print(score)
    print(doc.metadata["page"])
    print(doc.page_content[:200])
    print("-" * 60)